# Pelatihan Model & Evaluasi: DUCFF
Di sini kita memuat data preprocessed dan melatih model DUCFF raksasa.

In [5]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Dropout, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)


## Arsitektur DUCFF

In [6]:
inputs = Input(shape=input_shape)
x_tcn = tf.keras.layers.Conv1D(filters=128, kernel_size=2, padding='causal', activation='relu')(inputs)
x_tcn = tf.keras.layers.Conv1D(filters=64, kernel_size=2, padding='causal', activation='relu')(x_tcn)
x_tcn_flat = tf.keras.layers.Flatten()(x_tcn)
attention_output = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(inputs, inputs)
attention_output = LayerNormalization(epsilon=1e-6)(attention_output + inputs)
x_transformer_flat = tf.keras.layers.Flatten()(attention_output)
fusion = tf.keras.layers.Concatenate()([x_tcn_flat, x_transformer_flat])
x = Dense(256, activation='relu')(fusion)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(2, activation='linear')(x)
model = Model(inputs=inputs, outputs=outputs, name="DUCFF_Monster")
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "DUCFF_Monster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 10, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 2)     │      5,634 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 10, 128)   │        640 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 2)     │          0 │ multi_head_atten… │
│                     │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 10, 64)    │     16,448 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 2)     │          4 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 640)       │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 20)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 660)       │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    169,216 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │     16,448 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 2)         │        130 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 208,520 (814.53 KB)

 Trainable params: 208,520 (814.53 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_DUCFF.csv', append=True)
EPOCHS = 200
BATCH_SIZE = 64

print(f"\n--- Training: DUCFF ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)



--- Training: DUCFF ---
Epoch 1/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 0.0808 - mae: 0.2201 - val_loss: 0.0745 - val_mae: 0.2073
Epoch 2/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0738 - mae: 0.2102 - val_loss: 0.0714 - val_mae: 0.2027
Epoch 3/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0729 - mae: 0.2091 - val_loss: 0.0729 - val_mae: 0.2007
Epoch 4/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0721 - mae: 0.2080 - val_loss: 0.0721 - val_mae: 0.2064
Epoch 5/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0723 - mae: 0.2082 - val_loss: 0.0709 - val_mae: 0.1967
Epoch 6/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0716 - mae: 0.2071 - val_loss: 0.0716 - val_mae: 0.1969
Epoch 7/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0720 - mae: 0.2079 - val_loss: 0.0714 - val_mae: 0.1976
Epoch 8/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0709 - mae: 0.2059 - val_loss: 0.0724 - val_mae: 0.2048
Epoch 9/200
199

## Evaluasi Kecepatan & Akurasi

In [8]:
single_sample = X_test[0:1]

# Pemanasan prediksi
_ = model.predict(single_sample, verbose=0)

# Precise inference benchmark
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Prediksi seluruh data test
pred = model.predict(X_test, verbose=0)
np.save('pred_DUCFF.npy', pred)

results = {
    'DUCFF': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)


,MSE,RMSE,MAE,R_Squared (R²),Speed (ms)
DUCFF,0.074965,0.273798,0.211508,0.417984,64.14818
